# Home Credit Proxy Notebook 04 — Business-first + Product-ready Stage Modeling (A / B / C)

Mục tiêu của notebook này:

- train **3 model riêng** cho 3 stage `A / B / C`
- ưu tiên **business correctness** hơn là chạy theo AUC
- xử lý lại đúng bài toán **A sparse block**
- tạo output đủ tốt để sang notebook 5:
  - artifact model
  - validation score file
  - score band table
  - business summary theo stage

## Business principles của notebook này

1. **Không đọc `predict_proba` như PD thật**  
   Ở notebook 4, output model được dùng chủ yếu như **risk ranking score**.

2. **Stage nào dùng feature của stage đó**  
   - A → `a_*`
   - B → `b_*`
   - C → `c_*`

3. **Feature phải pass business sanity trước khi train**
   - đủ coverage
   - không constant
   - không toàn null ở train
   - không sụp ở valid

4. **Review model theo business**
   - direction của feature
   - score band separation
   - monotonicity của bad rate
   - concentration của population


Bản product-ready bổ sung thêm:
- score chuẩn hóa cho product (`score_300_900`)
- reason codes để show trên UI / admin review
- policy mode theo stage (`full_decision` vs `guardrail_only`)
- riêng Stage A được diễn giải như **starter-offer / onboarding recommendation model**,
  không phải full approval model
- inference pack hợp nhất A/B/C để notebook 5 hoặc Streamlit gọi trực tiếp
- product manifest / sample payload để gắn vào app

In [1]:
# =========================================================
# 0. Imports
# =========================================================
from __future__ import annotations

import gc
import json
import math
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(50)


polars.config.Config

In [2]:
# =========================================================
# 1. Paths
# =========================================================
INPUT_DIR_NB3 = Path("/kaggle/input/notebooks/hongvittrnh/notebook-3-latest/homecredit_proxy_notebook_03")

if not INPUT_DIR_NB3.exists():
    candidate_roots = [
        Path("/kaggle/working/homecredit_proxy_notebook_03"),
        Path("/kaggle/working"),
        Path("/mnt/data"),
        Path.cwd(),
    ]
    for p in candidate_roots:
        if (p / "proxy_features_A.parquet").exists() and (p / "proxy_features_B.parquet").exists() and (p / "proxy_features_C.parquet").exists():
            INPUT_DIR_NB3 = p
            break

WORK_DIR = Path("/kaggle/working/homecredit_proxy_notebook_04_business")
WORK_DIR.mkdir(parents=True, exist_ok=True)

A_PATH = INPUT_DIR_NB3 / "proxy_features_A.parquet"
B_PATH = INPUT_DIR_NB3 / "proxy_features_B.parquet"
C_PATH = INPUT_DIR_NB3 / "proxy_features_C.parquet"
PROXY_MART_PATH = INPUT_DIR_NB3 / "proxy_mart.parquet"

print("INPUT_DIR_NB3:", INPUT_DIR_NB3)
print("WORK_DIR     :", WORK_DIR)
print("A exists     :", A_PATH.exists())
print("B exists     :", B_PATH.exists())
print("C exists     :", C_PATH.exists())
print("proxy_mart   :", PROXY_MART_PATH.exists())


INPUT_DIR_NB3: /kaggle/input/notebooks/hongvittrnh/notebook-3-latest/homecredit_proxy_notebook_03
WORK_DIR     : /kaggle/working/homecredit_proxy_notebook_04_business
A exists     : True
B exists     : True
C exists     : True
proxy_mart   : False


In [3]:
# =========================================================
# 2. Load datasets
# =========================================================
A_df = pl.read_parquet(A_PATH)
B_df = pl.read_parquet(B_PATH)
C_df = pl.read_parquet(C_PATH)
proxy_mart = pl.read_parquet(PROXY_MART_PATH) if PROXY_MART_PATH.exists() else None

print("A_df:", A_df.shape)
print("B_df:", B_df.shape)
print("C_df:", C_df.shape)
if proxy_mart is not None:
    print("proxy_mart:", proxy_mart.shape)

display(A_df.head(3))
display(B_df.head(3))
display(C_df.head(3))


A_df: (318617, 57)
B_df: (1085030, 57)
C_df: (123012, 57)


case_id,decision_date,target,tenure_months_proxy,stage_final,stage_group,train_other_1.csv__amtdepositincoming_4809444a__max,a_amtdepositincoming_max,train_other_1.csv__amtdepositbalance_4809441a__max,a_amtdepositbalance_max,train_tax_registry_a_1.csv__amount_4527230a__max,a_tax_amount_4527230_max,train_tax_registry_b_1.csv__amount_4917619a__max,a_other_amount_4917619_max,train_static_cb_0.csv__contractssum_5085716l__max,a_contractssum_max,train_credit_bureau_b_1.csv__credquantity_1099l__max,a_credquantity_max,train_credit_bureau_b_1.csv__amount_1115a__max,a_amount_1115_max,train_static_0_0.csv__applicationcnt_361l__max,train_static_0_1.csv__applicationcnt_361l__max,a_applicationcnt_max,a_external_debt_to_incoming_ratio,a_deposit_balance_to_incoming_ratio,a_has_multi_application_signal,a_has_external_credit_exposure,train_applprev_1_0.csv__actualdpd_943p__max,train_applprev_1_1.csv__actualdpd_943p__max,b_actualdpd_max,train_static_0_0.csv__avgmaxdpdlast9m_3716943p__max,train_static_0_1.csv__avgmaxdpdlast9m_3716943p__max,b_avgmaxdpdlast9m_max,train_static_0_0.csv__amtinstpaidbefduel24m_4187115a__max,train_static_0_1.csv__amtinstpaidbefduel24m_4187115a__max,b_amtinstpaidbefdue_max,train_applprev_1_0.csv__annuity_853a__max,train_applprev_1_1.csv__annuity_853a__max,b_annuity_max,b_paid_before_due_to_annuity_ratio,b_has_recent_dpd,b_has_paid_before_due_signal,train_static_0_0.csv__avgdbddpdlast24m_3658932p__max,train_static_0_1.csv__avgdbddpdlast24m_3658932p__max,c_avgdbddpdlast24m_max,train_static_0_0.csv__avgdpdtolclosure24_3658938p__max,train_static_0_1.csv__avgdpdtolclosure24_3658938p__max,c_avgdpdtolclosure24_max,train_applprev_1_0.csv__actualdpd_943p__max_right,train_applprev_1_1.csv__actualdpd_943p__max_right,c_actualdpd_max,train_static_0_0.csv__datelastinstal40dpd_247d__max_date,train_static_0_1.csv__datelastinstal40dpd_247d__max_date,c_days_since_last_40dpd,c_recent_vs_long_dpd_ratio,c_has_active_delinquency_signal,c_recent_40dpd_flag
i64,date,f32,f32,str,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,date,date,f32,f32,f32,f32
873780,2019-11-24,0.0,null,"""A_new_no_internal_history""","""A""",null,null,null,null,5137.399902,5137.399902,null,null,null,null,null,null,null,null,0.0,null,0.0,null,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0
748264,2019-07-21,0.0,null,"""A_new_no_internal_history""","""A""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,0.0,null,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0
658207,2019-03-25,0.0,null,"""A_new_no_internal_history""","""A""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,0.0,null,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0


case_id,decision_date,target,tenure_months_proxy,stage_final,stage_group,train_other_1.csv__amtdepositincoming_4809444a__max,a_amtdepositincoming_max,train_other_1.csv__amtdepositbalance_4809441a__max,a_amtdepositbalance_max,train_tax_registry_a_1.csv__amount_4527230a__max,a_tax_amount_4527230_max,train_tax_registry_b_1.csv__amount_4917619a__max,a_other_amount_4917619_max,train_static_cb_0.csv__contractssum_5085716l__max,a_contractssum_max,train_credit_bureau_b_1.csv__credquantity_1099l__max,a_credquantity_max,train_credit_bureau_b_1.csv__amount_1115a__max,a_amount_1115_max,train_static_0_0.csv__applicationcnt_361l__max,train_static_0_1.csv__applicationcnt_361l__max,a_applicationcnt_max,a_external_debt_to_incoming_ratio,a_deposit_balance_to_incoming_ratio,a_has_multi_application_signal,a_has_external_credit_exposure,train_applprev_1_0.csv__actualdpd_943p__max,train_applprev_1_1.csv__actualdpd_943p__max,b_actualdpd_max,train_static_0_0.csv__avgmaxdpdlast9m_3716943p__max,train_static_0_1.csv__avgmaxdpdlast9m_3716943p__max,b_avgmaxdpdlast9m_max,train_static_0_0.csv__amtinstpaidbefduel24m_4187115a__max,train_static_0_1.csv__amtinstpaidbefduel24m_4187115a__max,b_amtinstpaidbefdue_max,train_applprev_1_0.csv__annuity_853a__max,train_applprev_1_1.csv__annuity_853a__max,b_annuity_max,b_paid_before_due_to_annuity_ratio,b_has_recent_dpd,b_has_paid_before_due_signal,train_static_0_0.csv__avgdbddpdlast24m_3658932p__max,train_static_0_1.csv__avgdbddpdlast24m_3658932p__max,c_avgdbddpdlast24m_max,train_static_0_0.csv__avgdpdtolclosure24_3658938p__max,train_static_0_1.csv__avgdpdtolclosure24_3658938p__max,c_avgdpdtolclosure24_max,train_applprev_1_0.csv__actualdpd_943p__max_right,train_applprev_1_1.csv__actualdpd_943p__max_right,c_actualdpd_max,train_static_0_0.csv__datelastinstal40dpd_247d__max_date,train_static_0_1.csv__datelastinstal40dpd_247d__max_date,c_days_since_last_40dpd,c_recent_vs_long_dpd_ratio,c_has_active_delinquency_signal,c_recent_40dpd_flag
i64,date,f32,f32,str,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,date,date,f32,f32,f32,f32
1483698,2019-08-10,0.0,27.530001,"""B_behavior_no_collection""","""B""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,0.0,null,null,0.0,0.0,0.0,null,0.0,null,null,null,21013.400391,null,21013.400391,8947.600586,null,8947.600586,2.348495,0.0,1.0,-2.0,null,-2.0,0.0,null,0.0,0.0,null,0.0,null,null,null,null,0.0,0.0
1538766,2019-09-15,0.0,20.440001,"""B_behavior_no_collection""","""B""",null,null,null,null,2415.0,2415.0,null,null,null,null,null,null,null,null,0.0,null,0.0,null,null,0.0,0.0,0.0,null,0.0,0.0,null,0.0,145964.0,null,145964.0,7298.200195,null,7298.200195,20.0,0.0,1.0,-2.0,null,-2.0,0.0,null,0.0,0.0,null,0.0,null,null,null,null,0.0,0.0
1395913,2019-06-09,0.0,5.29,"""B_behavior_no_collection""","""B""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,0.0,null,null,0.0,0.0,0.0,null,0.0,0.0,null,0.0,82530.0,null,82530.0,16506.0,null,16506.0,5.0,0.0,1.0,-10.0,null,-10.0,0.0,null,0.0,0.0,null,0.0,null,null,null,null,0.0,0.0


case_id,decision_date,target,tenure_months_proxy,stage_final,stage_group,train_other_1.csv__amtdepositincoming_4809444a__max,a_amtdepositincoming_max,train_other_1.csv__amtdepositbalance_4809441a__max,a_amtdepositbalance_max,train_tax_registry_a_1.csv__amount_4527230a__max,a_tax_amount_4527230_max,train_tax_registry_b_1.csv__amount_4917619a__max,a_other_amount_4917619_max,train_static_cb_0.csv__contractssum_5085716l__max,a_contractssum_max,train_credit_bureau_b_1.csv__credquantity_1099l__max,a_credquantity_max,train_credit_bureau_b_1.csv__amount_1115a__max,a_amount_1115_max,train_static_0_0.csv__applicationcnt_361l__max,train_static_0_1.csv__applicationcnt_361l__max,a_applicationcnt_max,a_external_debt_to_incoming_ratio,a_deposit_balance_to_incoming_ratio,a_has_multi_application_signal,a_has_external_credit_exposure,train_applprev_1_0.csv__actualdpd_943p__max,train_applprev_1_1.csv__actualdpd_943p__max,b_actualdpd_max,train_static_0_0.csv__avgmaxdpdlast9m_3716943p__max,train_static_0_1.csv__avgmaxdpdlast9m_3716943p__max,b_avgmaxdpdlast9m_max,train_static_0_0.csv__amtinstpaidbefduel24m_4187115a__max,train_static_0_1.csv__amtinstpaidbefduel24m_4187115a__max,b_amtinstpaidbefdue_max,train_applprev_1_0.csv__annuity_853a__max,train_applprev_1_1.csv__annuity_853a__max,b_annuity_max,b_paid_before_due_to_annuity_ratio,b_has_recent_dpd,b_has_paid_before_due_signal,train_static_0_0.csv__avgdbddpdlast24m_3658932p__max,train_static_0_1.csv__avgdbddpdlast24m_3658932p__max,c_avgdbddpdlast24m_max,train_static_0_0.csv__avgdpdtolclosure24_3658938p__max,train_static_0_1.csv__avgdpdtolclosure24_3658938p__max,c_avgdpdtolclosure24_max,train_applprev_1_0.csv__actualdpd_943p__max_right,train_applprev_1_1.csv__actualdpd_943p__max_right,c_actualdpd_max,train_static_0_0.csv__datelastinstal40dpd_247d__max_date,train_static_0_1.csv__datelastinstal40dpd_247d__max_date,c_days_since_last_40dpd,c_recent_vs_long_dpd_ratio,c_has_active_delinquency_signal,c_recent_40dpd_flag
i64,date,f32,f32,str,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,date,date,f32,f32,f32,f32
1804093,2020-03-03,0.0,null,"""C_collection_signal""","""C""",null,null,null,null,17681.400391,17681.400391,null,null,null,null,null,null,null,null,null,0.0,0.0,null,null,0.0,0.0,null,0.0,0.0,null,0.0,0.0,null,2270.400146,2270.400146,null,5161.0,5161.0,0.439915,0.0,1.0,null,1300.0,1300.0,null,1300.0,1300.0,null,0.0,0.0,null,2017-12-27,797.0,1.0,1.0,0.0
2642631,2019-11-19,0.0,61.830002,"""C_collection_signal""","""C""",null,null,null,null,3033.800049,3033.800049,null,null,null,null,null,null,null,null,0.0,null,0.0,null,null,0.0,0.0,0.0,null,0.0,0.0,null,0.0,23836.201172,null,23836.201172,2167.199951,null,2167.199951,10.998616,0.0,1.0,1.0,null,1.0,0.0,null,0.0,0.0,null,0.0,null,null,null,null,1.0,0.0
1887094,2020-08-05,0.0,25.76,"""C_collection_signal""","""C""",null,null,null,null,null,null,null,null,288422.125,288422.125,1.0,1.0,31255.0,31255.0,null,0.0,0.0,null,null,0.0,1.0,null,0.0,0.0,null,2.0,2.0,null,3184.0,3184.0,null,5217.399902,5217.399902,0.610266,1.0,1.0,null,2.0,2.0,null,2.0,2.0,null,0.0,0.0,null,2020-06-03,63.0,1.0,1.0,1.0


In [4]:
# =========================================================
# 3. Constants & stage configs
# =========================================================
CASE_COL = "case_id"
DATE_COL = "decision_date"
TARGET_COL = "target"
STAGE_COL = "stage_group"

BASE_KEEP_COLS = [
    CASE_COL, DATE_COL, TARGET_COL, STAGE_COL,
    "stage_final", "tenure_months_proxy", "anchor_quality_flag"
]

MODEL_CONFIG = {
    "A": {
        "prefix": "a_",
        "min_non_null_rate": 0.02,
        "max_missing_rate": 0.98,
        "min_unique_non_null": 2,
        "min_non_null_train": 500,
        "min_non_null_valid": 100,
        "max_dominant_share": 0.995,
        "C": 0.7,
        "max_iter": 1000,
        "valid_frac": 0.20,
    },
    "B": {
        "prefix": "b_",
        "min_non_null_rate": 0.03,
        "max_missing_rate": 0.97,
        "min_unique_non_null": 2,
        "min_non_null_train": 1000,
        "min_non_null_valid": 200,
        "max_dominant_share": 0.995,
        "C": 0.8,
        "max_iter": 1000,
        "valid_frac": 0.20,
    },
    "C": {
        "prefix": "c_",
        "min_non_null_rate": 0.03,
        "max_missing_rate": 0.97,
        "min_unique_non_null": 2,
        "min_non_null_train": 300,
        "min_non_null_valid": 80,
        "max_dominant_share": 0.995,
        "C": 0.8,
        "max_iter": 1200,
        "valid_frac": 0.20,
    },
}


# =========================================================
# 3.1 Product scoring config
# =========================================================
PRODUCT_CONFIG = {
    "score_floor": 300.0,
    "score_ceiling": 900.0,
    "reason_code_top_n": 3,
    "strict_min_features": {"A": 4, "B": 5, "C": 5},
    "guardrail_min_features": {"A": 2, "B": 4, "C": 4},
    "strict_max_band_breaks": {"A": 2, "B": 2, "C": 2},
    "score_label": "score_300_900",
}

STAGE_PRODUCT_POLICY = {
    "A": {
        "stage_name": "Application / New-to-bank Starter Offer",
        "default_mode": "starter_offer_only",
        "recommended_use": "starter loan recommendation + manual review support",
    },
    "B": {
        "stage_name": "Behavior / Thin-file",
        "default_mode": "full_decision",
        "recommended_use": "ranking + cutoff exploration + review prioritization",
    },
    "C": {
        "stage_name": "Collection / Mature risk",
        "default_mode": "guardrail_only",
        "recommended_use": "high-risk prioritization + collections treatment support",
    },
}


In [5]:
# =========================================================
# 4. Helpers
# =========================================================
def ensure_date(df: pl.DataFrame, col: str) -> pl.DataFrame:
    if col not in df.columns:
        return df
    dt = df.schema[col]
    if dt == pl.Date:
        return df
    if dt == pl.Datetime:
        return df.with_columns(pl.col(col).dt.date().alias(col))
    return df.with_columns(
        pl.col(col)
        .cast(pl.Utf8, strict=False)
        .str.strptime(pl.Date, strict=False)
        .alias(col)
    )

def cast_float64(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
    exprs = [pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in cols if c in df.columns]
    return df.with_columns(exprs) if exprs else df

def ks_statistic(y_true: np.ndarray, y_score: np.ndarray) -> float:
    order = np.argsort(y_score)
    y_true = y_true[order]
    total_pos = y_true.sum()
    total_neg = len(y_true) - total_pos
    if total_pos == 0 or total_neg == 0:
        return float("nan")
    cum_pos = np.cumsum(y_true) / total_pos
    cum_neg = np.cumsum(1 - y_true) / total_neg
    return float(np.max(np.abs(cum_pos - cum_neg)))

def safe_metric(fn, *args):
    try:
        return float(fn(*args))
    except Exception:
        return float("nan")

def make_time_split(df: pl.DataFrame, date_col: str, valid_frac: float = 0.2) -> tuple[pl.DataFrame, pl.DataFrame, str]:
    temp = df.clone()
    if date_col in temp.columns:
        temp = ensure_date(temp, date_col)
        if temp[date_col].null_count() < temp.height:
            temp = temp.sort([date_col, CASE_COL] if CASE_COL in temp.columns else [date_col])
            n_valid = max(1, int(round(temp.height * valid_frac)))
            n_train = max(1, temp.height - n_valid)
            train_df = temp.head(n_train)
            valid_df = temp.tail(n_valid)
            cutoff = valid_df[date_col].min() if valid_df.height > 0 else None
            return train_df, valid_df, f"time_split_last_{n_valid}_rows_cutoff_{cutoff}"

    temp = temp.sample(fraction=1.0, shuffle=True, seed=42)
    n_valid = max(1, int(round(temp.height * valid_frac)))
    n_train = max(1, temp.height - n_valid)
    return temp.head(n_train), temp.tail(n_valid), "random_fallback_split"

def feature_profile_from_df(df: pl.DataFrame, feature_cols: list[str]) -> pl.DataFrame:
    rows = []
    n_rows = max(1, df.height)
    for c in feature_cols:
        s = df[c]
        non_null = n_rows - s.null_count()
        non_null_rate = non_null / n_rows
        n_unique = s.drop_nulls().n_unique()
        dominant_share = None
        if non_null > 0:
            vc = (
                df.select(pl.col(c))
                .drop_nulls()
                .group_by(c)
                .len()
                .sort("len", descending=True)
            )
            dominant_share = float(vc["len"][0] / non_null) if vc.height > 0 else None
        rows.append({
            "feature": c,
            "n_rows": int(df.height),
            "non_null_count": int(non_null),
            "non_null_rate": float(non_null_rate),
            "missing_rate": float(1 - non_null_rate),
            "n_unique_non_null": int(n_unique),
            "dominant_share": float(dominant_share) if dominant_share is not None else None,
        })
    return pl.DataFrame(rows).sort(["non_null_rate", "feature"], descending=[True, False])

def pick_initial_stage_features(df: pl.DataFrame, stage: str) -> tuple[list[str], pl.DataFrame]:
    cfg = MODEL_CONFIG[stage]
    prefix = cfg["prefix"]
    candidate_cols = [c for c in df.columns if c.startswith(prefix)]
    if not candidate_cols:
        empty = pl.DataFrame({
            "feature": [], "n_rows": [], "non_null_count": [], "non_null_rate": [],
            "missing_rate": [], "n_unique_non_null": [], "dominant_share": []
        })
        return [], empty

    temp = cast_float64(df, candidate_cols)
    profile = feature_profile_from_df(temp, candidate_cols)

    keep = (
        profile
        .filter(
            (pl.col("non_null_rate") >= cfg["min_non_null_rate"]) &
            (pl.col("missing_rate") <= cfg["max_missing_rate"]) &
            (pl.col("n_unique_non_null") >= cfg["min_unique_non_null"]) &
            (
                pl.col("dominant_share").is_null() |
                (pl.col("dominant_share") <= cfg["max_dominant_share"])
            )
        )
        .sort(["non_null_rate", "feature"], descending=[True, False])
    )
    return keep["feature"].to_list(), profile

def prune_train_valid_stability(train_df: pl.DataFrame, valid_df: pl.DataFrame, feature_cols: list[str], stage: str) -> tuple[list[str], pl.DataFrame]:
    cfg = MODEL_CONFIG[stage]
    rows = []
    keep = []

    for c in feature_cols:
        tr = train_df[c]
        va = valid_df[c]

        tr_non_null = train_df.height - tr.null_count()
        va_non_null = valid_df.height - va.null_count()
        tr_unique = tr.drop_nulls().n_unique()
        va_unique = va.drop_nulls().n_unique()

        keep_flag = (
            tr_non_null >= cfg["min_non_null_train"] and
            va_non_null >= cfg["min_non_null_valid"] and
            tr_unique >= 2
        )

        rows.append({
            "feature": c,
            "train_non_null": int(tr_non_null),
            "valid_non_null": int(va_non_null),
            "train_unique_non_null": int(tr_unique),
            "valid_unique_non_null": int(va_unique),
            "kept_after_split": bool(keep_flag),
        })
        if keep_flag:
            keep.append(c)

    return keep, pl.DataFrame(rows).sort(["kept_after_split", "train_non_null", "feature"], descending=[True, True, False])

def to_numpy(df: pl.DataFrame, cols: list[str]) -> np.ndarray:
    if not cols:
        return np.empty((df.height, 0), dtype=float)
    return df.select([pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in cols]).to_numpy()

def make_band_table(scored_df: pl.DataFrame, score_col: str = "score_raw", n_bins: int = 10) -> pl.DataFrame:
    if scored_df.height == 0 or score_col not in scored_df.columns:
        return pl.DataFrame()

    pdf = scored_df.select([c for c in [CASE_COL, TARGET_COL, score_col] if c in scored_df.columns]).to_pandas()
    pdf = pdf.dropna(subset=[score_col]).copy()
    if pdf.empty:
        return pl.DataFrame()

    nunique = pdf[score_col].nunique()
    n_bins = int(min(n_bins, max(1, nunique)))

    if n_bins < 2:
        pdf["score_band"] = "Band_1"
    else:
        try:
            labels = [f"Band_{i+1}" for i in range(n_bins)]
            pdf["score_band"] = pd.qcut(pdf[score_col], q=n_bins, labels=labels, duplicates="drop")
        except Exception:
            ranks = pdf[score_col].rank(method="first")
            labels = [f"Band_{i+1}" for i in range(n_bins)]
            pdf["score_band"] = pd.qcut(ranks, q=n_bins, labels=labels, duplicates="drop")

    out = (
        pdf.groupby("score_band", observed=False)
        .agg(
            n_cases=(score_col, "size"),
            avg_score=(score_col, "mean"),
            bad_rate=(TARGET_COL, "mean"),
        )
        .reset_index()
        .sort_values("avg_score", ascending=False)
        .reset_index(drop=True)
    )
    out["population_share"] = out["n_cases"] / out["n_cases"].sum()
    overall_bad = pdf[TARGET_COL].mean()
    out["lift_vs_portfolio"] = out["bad_rate"] / overall_bad if overall_bad not in [0, np.nan] else np.nan
    out["bad_rate_diff_vs_portfolio"] = out["bad_rate"] - overall_bad
    out["band_rank"] = np.arange(1, len(out) + 1)
    out["is_monotonic_non_increasing"] = out["bad_rate"].is_monotonic_decreasing
    return pl.from_pandas(out)

def monotonicity_summary(band_table: pl.DataFrame) -> dict:
    if band_table.height <= 1 or "bad_rate" not in band_table.columns:
        return {"is_monotonic_non_increasing": None, "n_breaks": None}
    br = band_table["bad_rate"].to_list()
    breaks = 0
    for i in range(len(br) - 1):
        if br[i] < br[i + 1]:
            breaks += 1
    return {
        "is_monotonic_non_increasing": breaks == 0,
        "n_breaks": int(breaks),
    }

def describe_direction(coef: float) -> str:
    return "higher risk" if coef > 0 else "lower risk"

def build_business_interpretation(stage: str, coef_df: pl.DataFrame) -> pl.DataFrame:
    if coef_df.height == 0:
        return pl.DataFrame({
            "stage": [], "feature": [], "coefficient": [], "direction": [], "business_note": []
        })

    rows = []
    for row in coef_df.head(15).iter_rows(named=True):
        feature = row["feature"]
        coef = float(row["coefficient"])
        direction = describe_direction(coef)
        rows.append({
            "stage": stage,
            "feature": feature,
            "coefficient": coef,
            "direction": direction,
            "business_note": f"When {feature} increases, model reads this as {direction}."
        })
    return pl.DataFrame(rows)


def minmax_normalize_to_band(values: np.ndarray, floor: float = 300.0, ceiling: float = 900.0) -> np.ndarray:
    if len(values) == 0:
        return values
    vmin = float(np.nanmin(values))
    vmax = float(np.nanmax(values))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or math.isclose(vmin, vmax):
        return np.full_like(values, (floor + ceiling) / 2.0, dtype=float)
    scaled = (values - vmin) / (vmax - vmin)
    return floor + scaled * (ceiling - floor)

def make_reason_codes(scored_df: pl.DataFrame, coef_df: pl.DataFrame, stage: str, top_n: int = 3) -> pl.DataFrame:
    if scored_df.height == 0 or coef_df.height == 0:
        return scored_df
    stage_coef = coef_df.select(["feature", "coefficient"]).to_dicts()
    available = [r for r in stage_coef if r["feature"] in scored_df.columns]
    if len(available) == 0:
        return scored_df

    pdf = scored_df.select([c for c in scored_df.columns if c in {CASE_COL, TARGET_COL, "score_raw"} | set([r["feature"] for r in available])]).to_pandas()
    reason_cols = []
    for _, row in pdf.iterrows():
        contributions = []
        for item in available:
            feat = item["feature"]
            val = row.get(feat, np.nan)
            if pd.isna(val):
                continue
            contrib = float(val) * float(item["coefficient"])
            direction = "push_higher_risk" if contrib > 0 else "push_lower_risk"
            contributions.append((abs(contrib), feat, direction, float(val), float(item["coefficient"])))
        contributions = sorted(contributions, reverse=True)[:top_n]
        payload = [
            {
                "feature": feat,
                "direction": direction,
                "feature_value": round(val, 6),
                "coefficient": round(coef, 6),
                "contribution_proxy": round(abs_contrib, 6),
            }
            for abs_contrib, feat, direction, val, coef in contributions
        ]
        reason_cols.append(json.dumps(payload, ensure_ascii=False))
    return scored_df.with_columns(pl.Series("reason_codes_json", reason_cols))

def assign_stage_policy_mode(stage: str, metrics: dict) -> str:
    min_feat_strict = PRODUCT_CONFIG["strict_min_features"][stage]
    min_feat_guard = PRODUCT_CONFIG["guardrail_min_features"][stage]
    max_breaks_strict = PRODUCT_CONFIG["strict_max_band_breaks"][stage]
    n_feat = int(metrics.get("n_features_kept", 0))
    breaks = int(metrics.get("band_monotonic_breaks", 999) or 999)

    if stage == "A":
        if n_feat >= min_feat_guard:
            return "starter_offer_only"
        return "manual_review_only"

    if (n_feat >= min_feat_strict) and (breaks <= max_breaks_strict):
        return "full_decision"
    if n_feat >= min_feat_guard:
        return "guardrail_only"
    return "manual_review_only"

def make_product_bucket(score_expr: pl.Expr) -> pl.Expr:
    # Higher score = higher risk in this notebook because score_raw comes from model positive-class ordering.
    # Use quantile-style buckets via percent rank so this works inside with_columns(pl.col(...)).
    pct_rank = score_expr.cast(pl.Float64).rank("average").truediv(pl.len())
    return (
        pl.when(pct_rank >= 0.90).then(pl.lit("Very High Risk"))
        .when(pct_rank >= 0.75).then(pl.lit("High Risk"))
        .when(pct_rank >= 0.40).then(pl.lit("Medium Risk"))
        .otherwise(pl.lit("Low Risk"))
        .alias("risk_bucket")
    )

def make_product_action(stage: str, risk_bucket_expr: pl.Expr, policy_mode: str) -> pl.Expr:
    if policy_mode == "manual_review_only":
        return pl.lit("manual_review_required").alias("recommended_action")
    if stage == "A":
        return (
            pl.when(risk_bucket_expr == "Very High Risk").then(pl.lit("decline_or_manual_review"))
            .when(risk_bucket_expr == "High Risk").then(pl.lit("manual_review"))
            .when(risk_bucket_expr == "Medium Risk").then(pl.lit("starter_loan_small"))
            .otherwise(pl.lit("starter_loan_standard"))
            .alias("recommended_action")
        )
    if stage == "C":
        return (
            pl.when(risk_bucket_expr == "Very High Risk").then(pl.lit("priority_collection_queue"))
            .when(risk_bucket_expr == "High Risk").then(pl.lit("intensive_review"))
            .when(risk_bucket_expr == "Medium Risk").then(pl.lit("standard_review"))
            .otherwise(pl.lit("monitor"))
            .alias("recommended_action")
        )
    return (
        pl.when(risk_bucket_expr == "Very High Risk").then(pl.lit("decline_or_manual_review"))
        .when(risk_bucket_expr == "High Risk").then(pl.lit("manual_review"))
        .when(risk_bucket_expr == "Medium Risk").then(pl.lit("standard_review"))
        .otherwise(pl.lit("low_risk_candidate"))
        .alias("recommended_action")
    )


In [6]:
# =========================================================
# 5. Train one stage model
# =========================================================
def train_stage_model(df: pl.DataFrame, stage: str) -> dict:
    cfg = MODEL_CONFIG[stage]
    df = ensure_date(df, DATE_COL)

    if TARGET_COL not in df.columns:
        raise ValueError(f"{stage}: missing target column")
    if df.height == 0:
        raise ValueError(f"{stage}: empty dataframe")

    # stage feature scan
    initial_features, initial_profile = pick_initial_stage_features(df, stage)
    if len(initial_features) == 0:
        raise ValueError(f"{stage}: no usable features after initial screening")

    # keep only ids + target + features
    selected_cols = [c for c in BASE_KEEP_COLS if c in df.columns] + initial_features
    model_df = (
        df
        .select(selected_cols)
        .filter(pl.col(TARGET_COL).is_not_null())
    )

    # split first, then prune unstable features using train/valid only
    train_df, valid_df, split_note = make_time_split(model_df, DATE_COL, valid_frac=cfg["valid_frac"])

    train_df = cast_float64(train_df, initial_features)
    valid_df = cast_float64(valid_df, initial_features)

    kept_features, stability_df = prune_train_valid_stability(train_df, valid_df, initial_features, stage)
    if len(kept_features) == 0:
        raise ValueError(f"{stage}: all candidate features failed train/valid stability checks")

    X_train = to_numpy(train_df, kept_features)
    X_valid = to_numpy(valid_df, kept_features)
    y_train = train_df[TARGET_COL].cast(pl.Int8).to_numpy()
    y_valid = valid_df[TARGET_COL].cast(pl.Int8).to_numpy()

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=cfg["C"],
            max_iter=cfg["max_iter"],
            class_weight=None,
            solver="liblinear",
            random_state=42,
        )),
    ])
    pipe.fit(X_train, y_train)

    model = pipe.named_steps["model"]
    coef = model.coef_.ravel()
    active_mask = np.abs(coef) > 1e-12

    active_features = [f for f, keep in zip(kept_features, active_mask) if keep]
    active_coef = coef[active_mask]

    if len(active_features) == 0:
        active_features = kept_features
        active_coef = coef

    pred_train = pipe.predict_proba(X_train)[:, 1]
    pred_valid = pipe.predict_proba(X_valid)[:, 1]
    raw_train = pipe.decision_function(X_train)
    raw_valid = pipe.decision_function(X_valid)

    metrics = {
        "stage": stage,
        "n_rows_total": int(model_df.height),
        "n_rows_train": int(train_df.height),
        "n_rows_valid": int(valid_df.height),
        "n_features_input": int(len([c for c in df.columns if c.startswith(cfg["prefix"])])),
        "n_features_after_screen": int(len(initial_features)),
        "n_features_kept": int(len(kept_features)),
        "n_features_non_zero_coef": int(len(active_features)),
        "target_rate_total": float(model_df[TARGET_COL].mean()),
        "target_rate_train": float(train_df[TARGET_COL].mean()),
        "target_rate_valid": float(valid_df[TARGET_COL].mean()),
        "split_note": split_note,
        "auc_train": safe_metric(roc_auc_score, y_train, pred_train),
        "auc_valid": safe_metric(roc_auc_score, y_valid, pred_valid),
        "pr_auc_train": safe_metric(average_precision_score, y_train, pred_train),
        "pr_auc_valid": safe_metric(average_precision_score, y_valid, pred_valid),
        "ks_train": safe_metric(ks_statistic, y_train, pred_train),
        "ks_valid": safe_metric(ks_statistic, y_valid, pred_valid),
        "brier_train": safe_metric(brier_score_loss, y_train, pred_train),
        "brier_valid": safe_metric(brier_score_loss, y_valid, pred_valid),
        "logloss_train": safe_metric(log_loss, y_train, pred_train),
        "logloss_valid": safe_metric(log_loss, y_valid, pred_valid),
        "avg_pred_train": float(np.mean(pred_train)),
        "avg_pred_valid": float(np.mean(pred_valid)),
    }

    coef_df = (
        pl.DataFrame({
            "feature": active_features,
            "coefficient": active_coef,
            "abs_coefficient": np.abs(active_coef),
        })
        .sort("abs_coefficient", descending=True)
        .with_columns(
            pl.when(pl.col("coefficient") > 0)
            .then(pl.lit("higher risk"))
            .otherwise(pl.lit("lower risk"))
            .alias("direction")
        )
    )

    train_scored = (
        train_df
        .select([c for c in [CASE_COL, DATE_COL, TARGET_COL, STAGE_COL, "stage_final"] + kept_features if c in train_df.columns])
        .with_columns([
            pl.Series(name="score_raw", values=raw_train),
            pl.Series(name="score_proba", values=pred_train),
            pl.Series(
                name=PRODUCT_CONFIG["score_label"],
                values=minmax_normalize_to_band(
                    raw_train,
                    PRODUCT_CONFIG["score_floor"],
                    PRODUCT_CONFIG["score_ceiling"],
                ),
            ),
            pl.lit(stage).alias("model_stage"),
            pl.lit(stage).alias("stage"),
            pl.lit("train").alias("sample_type"),
        ])
    )

    valid_scored = (
        valid_df
        .select([c for c in [CASE_COL, DATE_COL, TARGET_COL, STAGE_COL, "stage_final"] + kept_features if c in valid_df.columns])
        .with_columns([
            pl.Series(name="score_raw", values=raw_valid),
            pl.Series(name="score_proba", values=pred_valid),
            pl.Series(
                name=PRODUCT_CONFIG["score_label"],
                values=minmax_normalize_to_band(
                    raw_valid,
                    PRODUCT_CONFIG["score_floor"],
                    PRODUCT_CONFIG["score_ceiling"],
                ),
            ),
            pl.lit(stage).alias("model_stage"),
            pl.lit(stage).alias("stage"),
            pl.lit("valid").alias("sample_type"),
        ])
    )

    band_table = make_band_table(valid_scored, score_col="score_raw", n_bins=10)
    mono = monotonicity_summary(band_table)
    metrics["band_monotonic_non_increasing"] = mono["is_monotonic_non_increasing"]
    metrics["band_monotonic_breaks"] = mono["n_breaks"]

    business_interpret = build_business_interpretation(stage, coef_df)

    return {
        "stage": stage,
        "config": cfg,
        "pipeline": pipe,
        "feature_profile_initial": initial_profile,
        "feature_stability": stability_df,
        "features_screened": initial_features,
        "features_kept": kept_features,
        "coef_df": coef_df,
        "train_scored": train_scored,
        "valid_scored": valid_scored,
        "band_table": band_table,
        "business_interpret": business_interpret,
        "metrics": metrics,
    }


In [7]:
# =========================================================
# 6. Pre-train sanity review
# =========================================================
for stage_name, df in {"A": A_df, "B": B_df, "C": C_df}.items():
    raw_stage_features = [c for c in df.columns if c.startswith(MODEL_CONFIG[stage_name]["prefix"])]
    target_rate = float(df[TARGET_COL].mean()) if TARGET_COL in df.columns else float("nan")
    print(
        f"[{stage_name}] rows={df.height:,} | raw_stage_features={len(raw_stage_features)} | "
        f"target_rate={target_rate:.4f}"
    )


[A] rows=318,617 | raw_stage_features=12 | target_rate=0.0226
[B] rows=1,085,030 | raw_stage_features=7 | target_rate=0.0278
[C] rows=123,012 | raw_stage_features=7 | target_rate=0.0863


In [8]:
# =========================================================
# 7. Train A / B / C
# =========================================================
results = {}
errors = {}

for stage_name, stage_df in {"A": A_df, "B": B_df, "C": C_df}.items():
    print("=" * 90)
    print(f"[RUN] Stage {stage_name}")
    try:
        obj = train_stage_model(stage_df, stage_name)
        results[stage_name] = obj
        print(
            f"[OK] Stage {stage_name} | kept={len(obj['features_kept'])} | "
            f"non_zero={obj['metrics']['n_features_non_zero_coef']} | "
            f"valid_rows={obj['metrics']['n_rows_valid']:,} | "
            f"monotonic={obj['metrics']['band_monotonic_non_increasing']}"
        )
    except Exception as e:
        errors[stage_name] = str(e)
        print(f"[ERROR] Stage {stage_name} failed: {e}")

print("\nFinished.")
print("Successful stages:", list(results.keys()))
print("Errors:", errors)


[RUN] Stage A
[OK] Stage A | kept=2 | non_zero=2 | valid_rows=63,723 | monotonic=False
[RUN] Stage B
[OK] Stage B | kept=6 | non_zero=6 | valid_rows=217,006 | monotonic=False
[RUN] Stage C
[OK] Stage C | kept=6 | non_zero=6 | valid_rows=24,602 | monotonic=False

Finished.
Successful stages: ['A', 'B', 'C']
Errors: {}


In [9]:
# =========================================================
# 8. Metrics summary
# =========================================================
metrics_df = (
    pl.DataFrame([obj["metrics"] for obj in results.values()])
    .sort("stage")
    if results else
    pl.DataFrame()
)
display(metrics_df)


stage,n_rows_total,n_rows_train,n_rows_valid,n_features_input,n_features_after_screen,n_features_kept,n_features_non_zero_coef,target_rate_total,target_rate_train,target_rate_valid,split_note,auc_train,auc_valid,pr_auc_train,pr_auc_valid,ks_train,ks_valid,brier_train,brier_valid,logloss_train,logloss_valid,avg_pred_train,avg_pred_valid,band_monotonic_non_increasing,band_monotonic_breaks
str,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,i64
"""A""",318617,254894,63723,12,4,2,2,0.022623,0.022974,0.021217,"""time_split_last_63723_rows_cut…",0.553017,0.546369,0.028048,0.026586,0.061178,0.122847,0.022411,0.020754,0.108593,0.102837,0.022996,0.023039,false,3
"""B""",1085030,868024,217006,7,6,6,6,0.027805,0.029262,0.021976,"""time_split_last_217006_rows_cu…",0.604548,0.627878,0.042248,0.034724,0.165529,0.22536,0.02831,0.021444,0.130476,0.104416,0.029267,0.027614,false,2
"""C""",123012,98410,24602,7,6,6,6,0.086309,0.089584,0.073205,"""time_split_last_24602_rows_cut…",0.594965,0.627189,0.129732,0.106594,0.171663,0.23954,0.080956,0.068157,0.298422,0.262272,0.089615,0.092633,false,5


In [10]:
# =========================================================
# 9. Feature review
# =========================================================
feature_profile_rows = []
feature_stability_rows = []

for stage_name, obj in results.items():
    feature_profile_rows.append(
        obj["feature_profile_initial"].with_columns(pl.lit(stage_name).alias("stage"))
    )
    feature_stability_rows.append(
        obj["feature_stability"].with_columns(pl.lit(stage_name).alias("stage"))
    )

feature_profile_df = pl.concat(feature_profile_rows, how="diagonal_relaxed") if feature_profile_rows else pl.DataFrame()
feature_stability_df = pl.concat(feature_stability_rows, how="diagonal_relaxed") if feature_stability_rows else pl.DataFrame()

display(feature_profile_df)
display(feature_stability_df)


feature,n_rows,non_null_count,non_null_rate,missing_rate,n_unique_non_null,dominant_share,stage
str,i64,i64,f64,f64,i64,f64,str
"""a_applicationcnt_max""",318617,318617,1.0,0.0,5,0.999978,"""A"""
"""a_has_external_credit_exposure""",318617,318617,1.0,0.0,2,0.981278,"""A"""
"""a_has_multi_application_signal""",318617,318617,1.0,0.0,2,0.999984,"""A"""
"""a_tax_amount_4527230_max""",318617,97266,0.305276,0.694724,34706,0.097855,"""A"""
"""a_other_amount_4917619_max""",318617,23773,0.074613,0.925387,15883,0.056282,"""A"""
"""a_contractssum_max""",318617,16861,0.052919,0.947081,12285,0.264694,"""A"""
"""a_credquantity_max""",318617,5965,0.018722,0.981278,9,0.705281,"""A"""
"""a_amount_1115_max""",318617,2454,0.007702,0.992298,1458,0.030155,"""A"""
"""a_amtdepositbalance_max""",318617,597,0.001874,0.998126,429,0.273032,"""A"""


feature,train_non_null,valid_non_null,train_unique_non_null,valid_unique_non_null,kept_after_split,stage
str,i64,i64,i64,i64,bool,str
"""a_has_external_credit_exposure""",254894,63723,2,2,true,"""A"""
"""a_tax_amount_4527230_max""",75559,21707,30182,12261,true,"""A"""
"""a_contractssum_max""",0,16861,0,12285,false,"""A"""
"""a_other_amount_4917619_max""",0,23773,0,15883,false,"""A"""
"""b_has_paid_before_due_signal""",868024,217006,2,2,true,"""B"""
"""b_has_recent_dpd""",868024,217006,2,2,true,"""B"""
"""b_annuity_max""",865748,216619,77806,58123,true,"""B"""
"""b_amtinstpaidbefdue_max""",660018,194696,444678,155101,true,"""B"""
"""b_paid_before_due_to_annuity_r…",646780,191873,455071,148588,true,"""B"""


In [11]:
# =========================================================
# 10. Top business drivers by stage
# =========================================================
for stage_name, obj in results.items():
    print(f"\n[TOP DRIVERS] Stage {stage_name}")
    display(obj["coef_df"].head(20))



[TOP DRIVERS] Stage A


feature,coefficient,abs_coefficient,direction
str,f64,f64,str
"""a_tax_amount_4527230_max""",-0.498554,0.498554,"""lower risk"""
"""a_has_external_credit_exposure""",0.096841,0.096841,"""higher risk"""



[TOP DRIVERS] Stage B


feature,coefficient,abs_coefficient,direction
str,f64,f64,str
"""b_has_recent_dpd""",0.215971,0.215971,"""higher risk"""
"""b_has_paid_before_due_signal""",-0.14449,0.14449,"""lower risk"""
"""b_amtinstpaidbefdue_max""",-0.130382,0.130382,"""lower risk"""
"""b_paid_before_due_to_annuity_r…",-0.077872,0.077872,"""lower risk"""
"""b_annuity_max""",0.060718,0.060718,"""higher risk"""
"""b_avgmaxdpdlast9m_max""",0.034761,0.034761,"""higher risk"""



[TOP DRIVERS] Stage C


feature,coefficient,abs_coefficient,direction
str,f64,f64,str
"""c_avgdbddpdlast24m_max""",0.294416,0.294416,"""higher risk"""
"""c_recent_40dpd_flag""",0.166611,0.166611,"""higher risk"""
"""c_days_since_last_40dpd""",-0.138341,0.138341,"""lower risk"""
"""c_avgdpdtolclosure24_max""",-0.129982,0.129982,"""lower risk"""
"""c_recent_vs_long_dpd_ratio""",-0.104551,0.104551,"""lower risk"""
"""c_actualdpd_max""",-0.006495,0.006495,"""lower risk"""


In [12]:
# =========================================================
# 11. Validation score bands
# =========================================================
for stage_name, obj in results.items():
    print(f"\n[SCORE BANDS] Stage {stage_name}")
    display(obj["band_table"])



[SCORE BANDS] Stage A


score_band,n_cases,avg_score,bad_rate,population_share,lift_vs_portfolio,bad_rate_diff_vs_portfolio,band_rank,is_monotonic_non_increasing
cat,i64,f64,f32,f64,f32,f32,i64,bool
"""Band_10""",6373,-3.367438,0.035776,0.100011,1.686205,0.014559,1,false
"""Band_9""",6372,-3.647808,0.03076,0.099995,1.449772,0.009543,2,false
"""Band_8""",6372,-3.74336,0.016478,0.099995,0.776664,-0.004738,3,false
"""Band_6""",6372,-3.74336,0.014438,0.099995,0.680505,-0.006779,4,false
"""Band_3""",6372,-3.74336,0.023227,0.099995,1.094726,0.00201,5,false
"""Band_4""",6372,-3.74336,0.01334,0.099995,0.628728,-0.007877,6,false
"""Band_7""",6372,-3.74336,0.011456,0.099995,0.539966,-0.00976,7,false
"""Band_5""",6373,-3.74336,0.018045,0.100011,0.850498,-0.003172,8,false
"""Band_2""",6372,-3.826941,0.029347,0.099995,1.383201,0.00813,9,false



[SCORE BANDS] Stage B


score_band,n_cases,avg_score,bad_rate,population_share,lift_vs_portfolio,bad_rate_diff_vs_portfolio,band_rank,is_monotonic_non_increasing
cat,i64,f64,f32,f64,f32,f32,i64,bool
"""Band_10""",21701,-2.953579,0.036542,0.100002,1.662792,0.014566,1,false
"""Band_9""",21676,-3.275727,0.031463,0.099887,1.431692,0.009487,2,false
"""Band_8""",21725,-3.363142,0.033418,0.100112,1.520622,0.011441,3,false
"""Band_7""",21700,-3.507102,0.032811,0.099997,1.493017,0.010835,4,false
"""Band_6""",21701,-3.636525,0.022626,0.100002,1.029547,0.000649,5,false
"""Band_5""",21700,-3.69149,0.015346,0.099997,0.698279,-0.006631,6,false
"""Band_4""",21701,-3.751054,0.013179,0.100002,0.599695,-0.008797,7,false
"""Band_3""",21700,-3.830945,0.01212,0.099997,0.551494,-0.009857,8,false
"""Band_2""",21701,-3.962651,0.010691,0.100002,0.486466,-0.011286,9,false



[SCORE BANDS] Stage C


score_band,n_cases,avg_score,bad_rate,population_share,lift_vs_portfolio,bad_rate_diff_vs_portfolio,band_rank,is_monotonic_non_increasing
cat,i64,f64,f32,f64,f32,f32,i64,bool
"""Band_10""",2461,-1.565511,0.108086,0.100033,1.476477,0.034881,1,false
"""Band_9""",2460,-2.220706,0.12561,0.099992,1.715853,0.052404,2,false
"""Band_8""",2460,-2.325714,0.143089,0.099992,1.954628,0.069884,3,false
"""Band_7""",2046,-2.36803,0.077224,0.083164,1.054892,0.004018,4,false
"""Band_6""",2802,-2.385306,0.061028,0.113893,0.833652,-0.012178,5,false
"""Band_5""",2119,-2.39319,0.045304,0.086131,0.618866,-0.027901,6,false
"""Band_4""",1542,-2.393765,0.035019,0.062678,0.478372,-0.038186,7,false
"""Band_3""",3677,-2.399469,0.044058,0.149459,0.601836,-0.029148,8,false
"""Band_2""",2562,-2.444377,0.044887,0.104138,0.613162,-0.028319,9,false


In [13]:
# =========================================================
# 12. Business interpretation table
# =========================================================
business_interpret_df = (
    pl.concat([obj["business_interpret"] for obj in results.values()], how="diagonal_relaxed")
    if results else
    pl.DataFrame()
)
display(business_interpret_df)


stage,feature,coefficient,direction,business_note
str,str,f64,str,str
"""A""","""a_tax_amount_4527230_max""",-0.498554,"""lower risk""","""When a_tax_amount_4527230_max …"
"""A""","""a_has_external_credit_exposure""",0.096841,"""higher risk""","""When a_has_external_credit_exp…"
"""B""","""b_has_recent_dpd""",0.215971,"""higher risk""","""When b_has_recent_dpd increase…"
"""B""","""b_has_paid_before_due_signal""",-0.14449,"""lower risk""","""When b_has_paid_before_due_sig…"
"""B""","""b_amtinstpaidbefdue_max""",-0.130382,"""lower risk""","""When b_amtinstpaidbefdue_max i…"
"""B""","""b_paid_before_due_to_annuity_r…",-0.077872,"""lower risk""","""When b_paid_before_due_to_annu…"
"""B""","""b_annuity_max""",0.060718,"""higher risk""","""When b_annuity_max increases, …"
"""B""","""b_avgmaxdpdlast9m_max""",0.034761,"""higher risk""","""When b_avgmaxdpdlast9m_max inc…"
"""C""","""c_avgdbddpdlast24m_max""",0.294416,"""higher risk""","""When c_avgdbddpdlast24m_max in…"


In [14]:

# =========================================================
# 13. Business + product readiness review
# =========================================================
readiness_rows = []

for stage_name, obj in results.items():
    m = obj["metrics"]
    policy_mode = assign_stage_policy_mode(stage_name, m)
    readiness_rows.append({
        "stage": stage_name,
        "has_features": m["n_features_kept"] > 0,
        "has_valid_split": m["n_rows_valid"] > 0,
        "has_non_zero_coef": m["n_features_non_zero_coef"] > 0,
        "n_features_kept": m["n_features_kept"],
        "band_monotonic_non_increasing": m["band_monotonic_non_increasing"],
        "band_monotonic_breaks": m["band_monotonic_breaks"],
        "policy_mode": policy_mode,
        "ready_for_nb5_business_strict": (
            (m["n_features_kept"] >= PRODUCT_CONFIG["strict_min_features"][stage_name]) and
            (m["n_rows_valid"] > 0) and
            (m["n_features_non_zero_coef"] > 0) and
            (int(m["band_monotonic_breaks"] or 999) <= PRODUCT_CONFIG["strict_max_band_breaks"][stage_name])
        ),
        "ready_for_product_scoring_api": (
            (m["n_features_kept"] >= PRODUCT_CONFIG["guardrail_min_features"][stage_name]) and
            (m["n_rows_valid"] > 0) and
            (m["n_features_non_zero_coef"] > 0)
        ),
        "note": (
            (
                "Stage A should be used as starter-offer / onboarding recommendation. "
                "Use score_raw / score_300_900 for ranking; do not frame this as a full approval PD model. "
                if stage_name == "A" else
                "Use score_raw / score_300_900 for ranking. score_proba is not calibrated PD. "
            ) + f"Current mode for stage {stage_name}: {policy_mode}."
        )
    })

readiness_df = pl.DataFrame(readiness_rows).sort("stage") if readiness_rows else pl.DataFrame()
display(readiness_df)


stage,has_features,has_valid_split,has_non_zero_coef,n_features_kept,band_monotonic_non_increasing,band_monotonic_breaks,policy_mode,ready_for_nb5_business_strict,ready_for_product_scoring_api,note
str,bool,bool,bool,i64,bool,i64,str,bool,bool,str
"""A""",true,true,true,2,false,3,"""starter_offer_only""",false,true,"""Stage A should be used as star…"
"""B""",true,true,true,6,false,2,"""full_decision""",true,true,"""Use score_raw / score_300_900 …"
"""C""",true,true,true,6,false,5,"""guardrail_only""",false,true,"""Use score_raw / score_300_900 …"


In [15]:
# =========================================================
# 13.1 Build product scoring views
# =========================================================
product_views = {}
product_policy_rows = []

for stage_name, obj in results.items():
    policy_candidates = (
        readiness_df
        .filter(pl.col("stage") == stage_name)
        .select("policy_mode")
        .to_series()
        .to_list()
        if readiness_df.height > 0
        else []
    )
    policy_mode = (
        policy_candidates[0]
        if len(policy_candidates) > 0
        else STAGE_PRODUCT_POLICY[stage_name]["default_mode"]
    )

    score_label = PRODUCT_CONFIG["score_label"]
    base_scored = obj["valid_scored"]

    if score_label not in base_scored.columns:
        raw_values = base_scored["score_raw"].to_numpy()
        base_scored = base_scored.with_columns(
            pl.Series(
                name=score_label,
                values=minmax_normalize_to_band(
                    raw_values,
                    PRODUCT_CONFIG["score_floor"],
                    PRODUCT_CONFIG["score_ceiling"],
                ),
            )
        )

    scored = (
        base_scored
        .with_columns(make_product_bucket(pl.col(score_label)))
        .with_columns(make_product_action(stage_name, pl.col("risk_bucket"), policy_mode))
        .with_columns(pl.lit(policy_mode).alias("policy_mode"))
        .with_columns(pl.lit(STAGE_PRODUCT_POLICY[stage_name]["recommended_use"]).alias("recommended_use"))
    )

    scored = make_reason_codes(
        scored_df=scored,
        coef_df=obj["coef_df"],
        stage=stage_name,
        top_n=PRODUCT_CONFIG["reason_code_top_n"],
    )
    product_views[stage_name] = scored

    bucket_summary = (
        scored
        .group_by(["risk_bucket", "recommended_action"])
        .agg([
            pl.len().alias("n_cases"),
            pl.mean(TARGET_COL).alias("bad_rate"),
            pl.mean(PRODUCT_CONFIG["score_label"]).alias("avg_score_300_900"),
        ])
        .sort("avg_score_300_900", descending=True)
    )

    print(f"[PRODUCT VIEW] Stage {stage_name}")
    display(bucket_summary)

    for row in bucket_summary.to_dicts():
        row.update({
            "stage": stage_name,
            "policy_mode": policy_mode,
            "stage_name": STAGE_PRODUCT_POLICY[stage_name]["stage_name"],
            "recommended_use": STAGE_PRODUCT_POLICY[stage_name]["recommended_use"],
        })
        product_policy_rows.append(row)


[PRODUCT VIEW] Stage A


risk_bucket,recommended_action,n_cases,bad_rate,avg_score_300_900
str,str,u32,f32,f64
"""Very High Risk""","""decline_or_manual_review""",6376,0.035759,842.723169
"""High Risk""","""manual_review""",5453,0.033743,823.071542
"""Medium Risk""","""starter_loan_small""",41137,0.0167,814.777871
"""Low Risk""","""starter_loan_standard""",10757,0.02352,760.78498


[PRODUCT VIEW] Stage B


risk_bucket,recommended_action,n_cases,bad_rate,avg_score_300_900
str,str,u32,f32,f64
"""Very High Risk""","""decline_or_manual_review""",21701,0.036542,753.336979
"""High Risk""","""manual_review""",32551,0.029861,744.549159
"""Medium Risk""","""standard_review""",75952,0.025964,737.161583
"""Low Risk""","""low_risk_candidate""",86802,0.011889,727.52262


[PRODUCT VIEW] Stage C


risk_bucket,recommended_action,n_cases,bad_rate,avg_score_300_900
str,str,u32,f32,f64
"""Very High Risk""","""priority_collection_queue""",2461,0.108086,782.507357
"""High Risk""","""intensive_review""",3690,0.13523,728.990768
"""Medium Risk""","""standard_review""",9008,0.069938,719.236995
"""Low Risk""","""monitor""",9443,0.042995,710.021346


In [16]:
# =========================================================
# 13.2 Product manifest + unified inference package
# =========================================================
product_manifest = {
    "score_label": PRODUCT_CONFIG["score_label"],
    "score_range": [PRODUCT_CONFIG["score_floor"], PRODUCT_CONFIG["score_ceiling"]],
    "global_note": "Higher score means higher risk in current notebook. score_proba is not calibrated PD. Stage A is a starter-offer recommendation model, not a full approval PD model.",
    "stages": {},
}

inference_pack = {
    "global_note": "Unified stage scoring pack for notebook 5 / Streamlit / batch scoring",
    "score_label": PRODUCT_CONFIG["score_label"],
    "score_range": [PRODUCT_CONFIG["score_floor"], PRODUCT_CONFIG["score_ceiling"]],
    "stage_models": {},
}

for stage_name, obj in results.items():
    readiness_rows = readiness_df.filter(pl.col("stage") == stage_name).to_dicts()
    readiness_row = readiness_rows[0] if len(readiness_rows) > 0 else {
        "policy_mode": STAGE_PRODUCT_POLICY[stage_name]["default_mode"],
        "ready_for_nb5_business_strict": False,
        "ready_for_product_scoring_api": True,
    }

    product_manifest["stages"][stage_name] = {
        "stage_name": STAGE_PRODUCT_POLICY[stage_name]["stage_name"],
        "policy_mode": readiness_row["policy_mode"],
        "ready_for_nb5_business_strict": readiness_row["ready_for_nb5_business_strict"],
        "ready_for_product_scoring_api": readiness_row["ready_for_product_scoring_api"],
        "features_kept": obj["features_kept"],
        "n_features_kept": obj["metrics"]["n_features_kept"],
        "recommended_use": STAGE_PRODUCT_POLICY[stage_name]["recommended_use"],
        "score_note": "Prefer score_raw / score_300_900 for ranking and policy experiments. For stage A, use this for starter-offer recommendation rather than full approval.",
        "reason_code_note": "reason_codes_json contains top signed contribution proxies based on feature value * coefficient.",
    }

    inference_pack["stage_models"][stage_name] = {
        "stage": stage_name,
        "pipeline": obj["pipeline"],
        "features_kept": obj["features_kept"],
        "config": obj["config"],
        "policy_mode": readiness_row["policy_mode"],
        "score_label": PRODUCT_CONFIG["score_label"],
        "score_note": "score_proba is not calibrated PD",
    }

display(
    pl.DataFrame(product_policy_rows)
    .sort(["stage", "avg_score_300_900"], descending=[False, True])
)


risk_bucket,recommended_action,n_cases,bad_rate,avg_score_300_900,stage,policy_mode,stage_name,recommended_use
str,str,i64,f64,f64,str,str,str,str
"""Very High Risk""","""decline_or_manual_review""",6376,0.035759,842.723169,"""A""","""starter_offer_only""","""Application / New-to-bank Star…","""starter loan recommendation + …"
"""High Risk""","""manual_review""",5453,0.033743,823.071542,"""A""","""starter_offer_only""","""Application / New-to-bank Star…","""starter loan recommendation + …"
"""Medium Risk""","""starter_loan_small""",41137,0.0167,814.777871,"""A""","""starter_offer_only""","""Application / New-to-bank Star…","""starter loan recommendation + …"
"""Low Risk""","""starter_loan_standard""",10757,0.02352,760.78498,"""A""","""starter_offer_only""","""Application / New-to-bank Star…","""starter loan recommendation + …"
"""Very High Risk""","""decline_or_manual_review""",21701,0.036542,753.336979,"""B""","""full_decision""","""Behavior / Thin-file""","""ranking + cutoff exploration +…"
"""High Risk""","""manual_review""",32551,0.029861,744.549159,"""B""","""full_decision""","""Behavior / Thin-file""","""ranking + cutoff exploration +…"
"""Medium Risk""","""standard_review""",75952,0.025964,737.161583,"""B""","""full_decision""","""Behavior / Thin-file""","""ranking + cutoff exploration +…"
"""Low Risk""","""low_risk_candidate""",86802,0.011889,727.52262,"""B""","""full_decision""","""Behavior / Thin-file""","""ranking + cutoff exploration +…"
"""Very High Risk""","""priority_collection_queue""",2461,0.108086,782.507357,"""C""","""guardrail_only""","""Collection / Mature risk""","""high-risk prioritization + col…"


In [17]:

# =========================================================
# 13.3 Product sample payloads for UI / API
# =========================================================
sample_payload_rows = []

for stage_name, scored in product_views.items():
    sample = (
        scored
        .select([
            c for c in [
                CASE_COL, "stage", TARGET_COL, "score_raw", PRODUCT_CONFIG["score_label"],
                "risk_bucket", "recommended_action", "policy_mode", "reason_codes_json"
            ] if c in scored.columns
        ])
        .head(5)
    )
    print(f"[SAMPLE PAYLOAD] Stage {stage_name}")
    display(sample)
    sample_payload_rows.extend(sample.to_dicts())


[SAMPLE PAYLOAD] Stage A


case_id,stage,target,score_raw,score_300_900,risk_bucket,recommended_action,policy_mode,reason_codes_json
i64,str,f32,f64,f64,str,str,str,str
937112,"""A""",0.0,-3.74336,814.777871,"""Medium Risk""","""starter_loan_small""","""starter_offer_only""","""[{""feature"": ""a_has_external_c…"
937113,"""A""",0.0,-3.557784,828.576221,"""High Risk""","""manual_review""","""starter_offer_only""","""[{""feature"": ""a_tax_amount_452…"
937114,"""A""",0.0,-3.624755,823.596616,"""High Risk""","""manual_review""","""starter_offer_only""","""[{""feature"": ""a_tax_amount_452…"
937116,"""A""",0.0,-3.74336,814.777871,"""Medium Risk""","""starter_loan_small""","""starter_offer_only""","""[{""feature"": ""a_has_external_c…"
937117,"""A""",0.0,-3.74336,814.777871,"""Medium Risk""","""starter_loan_small""","""starter_offer_only""","""[{""feature"": ""a_has_external_c…"


[SAMPLE PAYLOAD] Stage B


case_id,stage,target,score_raw,score_300_900,risk_bucket,recommended_action,policy_mode,reason_codes_json
i64,str,f32,f64,f64,str,str,str,str
968063,"""B""",0.0,-3.410412,741.554795,"""Medium Risk""","""standard_review""","""full_decision""","""[{""feature"": ""b_annuity_max"", …"
968066,"""B""",0.0,-3.67334,734.773618,"""Medium Risk""","""standard_review""","""full_decision""","""[{""feature"": ""b_amtinstpaidbef…"
968068,"""B""",0.0,-3.416629,741.394442,"""Medium Risk""","""standard_review""","""full_decision""","""[{""feature"": ""b_annuity_max"", …"
968074,"""B""",0.0,-3.864861,729.834091,"""Low Risk""","""low_risk_candidate""","""full_decision""","""[{""feature"": ""b_amtinstpaidbef…"
968075,"""B""",0.0,-3.387811,742.137688,"""Medium Risk""","""standard_review""","""full_decision""","""[{""feature"": ""b_annuity_max"", …"


[SAMPLE PAYLOAD] Stage C


case_id,stage,target,score_raw,score_300_900,risk_bucket,recommended_action,policy_mode,reason_codes_json
i64,str,f32,f64,f64,str,str,str,str
1809603,"""C""",1.0,-2.240856,729.872093,"""High Risk""","""intensive_review""","""guardrail_only""","""[{""feature"": ""c_avgdbddpdlast2…"
1809606,"""C""",0.0,-2.473586,711.733493,"""Low Risk""","""monitor""","""guardrail_only""","""[{""feature"": ""c_avgdbddpdlast2…"
1809620,"""C""",0.0,-2.393946,717.940531,"""Low Risk""","""monitor""","""guardrail_only""","""[{""feature"": ""c_avgdbddpdlast2…"
1809623,"""C""",0.0,-2.606192,701.398468,"""Low Risk""","""monitor""","""guardrail_only""","""[{""feature"": ""c_days_since_las…"
1809639,"""C""",0.0,-2.394471,717.899631,"""Low Risk""","""monitor""","""guardrail_only""","""[{""feature"": ""c_avgdbddpdlast2…"


In [18]:

# =========================================================
# 14. Export artifacts
# =========================================================
saved_paths = []

if metrics_df.height > 0:
    p = WORK_DIR / "stage_model_metrics.csv"
    metrics_df.write_csv(p)
    saved_paths.append(str(p))

if feature_profile_df.height > 0:
    p = WORK_DIR / "stage_feature_profile.csv"
    feature_profile_df.write_csv(p)
    saved_paths.append(str(p))

if feature_stability_df.height > 0:
    p = WORK_DIR / "stage_feature_stability.csv"
    feature_stability_df.write_csv(p)
    saved_paths.append(str(p))

if business_interpret_df.height > 0:
    p = WORK_DIR / "stage_business_interpretation.csv"
    business_interpret_df.write_csv(p)
    saved_paths.append(str(p))

if readiness_df.height > 0:
    p = WORK_DIR / "stage_business_readiness.csv"
    readiness_df.write_csv(p)
    saved_paths.append(str(p))

if len(product_policy_rows) > 0:
    product_policy_df = pl.DataFrame(product_policy_rows)
    p = WORK_DIR / "stage_product_policy_summary.csv"
    product_policy_df.write_csv(p)
    saved_paths.append(str(p))

manifest_path = WORK_DIR / "product_scoring_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(product_manifest, f, ensure_ascii=False, indent=2)
saved_paths.append(str(manifest_path))

pack_path = WORK_DIR / "unified_stage_inference_pack.joblib"
joblib.dump(inference_pack, pack_path)
saved_paths.append(str(pack_path))

for stage_name, obj in results.items():
    model_path = WORK_DIR / f"stage_{stage_name.lower()}_model.joblib"
    joblib.dump({
        "stage": stage_name,
        "pipeline": obj["pipeline"],
        "features_kept": obj["features_kept"],
        "config": obj["config"],
        "score_label": PRODUCT_CONFIG["score_label"],
        "score_note": "score_raw / score_300_900 are for ranking; score_proba is not calibrated business PD",
    }, model_path)
    saved_paths.append(str(model_path))

    valid_score_path = WORK_DIR / f"stage_{stage_name.lower()}_valid_scored.parquet"
    obj["valid_scored"].write_parquet(valid_score_path)
    saved_paths.append(str(valid_score_path))

    if stage_name in product_views:
        product_score_path = WORK_DIR / f"stage_{stage_name.lower()}_product_scored.parquet"
        product_views[stage_name].write_parquet(product_score_path)
        saved_paths.append(str(product_score_path))

    band_path = WORK_DIR / f"stage_{stage_name.lower()}_band_table.csv"
    obj["band_table"].write_csv(band_path)
    saved_paths.append(str(band_path))

sample_payload_path = WORK_DIR / "product_sample_payload_rows.json"
with open(sample_payload_path, "w", encoding="utf-8") as f:
    json.dump(sample_payload_rows, f, ensure_ascii=False, indent=2, default=str)
saved_paths.append(str(sample_payload_path))

print("Saved artifacts:")
for p in saved_paths:
    print("-", p)


Saved artifacts:
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_model_metrics.csv
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_feature_profile.csv
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_feature_stability.csv
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_business_interpretation.csv
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_business_readiness.csv
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_product_policy_summary.csv
- /kaggle/working/homecredit_proxy_notebook_04_business/product_scoring_manifest.json
- /kaggle/working/homecredit_proxy_notebook_04_business/unified_stage_inference_pack.joblib
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_a_model.joblib
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_a_valid_scored.parquet
- /kaggle/working/homecredit_proxy_notebook_04_business/stage_a_product_scored.parquet
- /kaggle/working/homecredit_proxy_notebook_04

In [19]:

# =========================================================
# 15. Final recommendation for notebook 5 / product handoff
# =========================================================
if readiness_df.height == 0:
    print("[STOP] No successful stage model. Do not move to notebook 5.")
else:
    all_strict = bool(readiness_df["ready_for_nb5_business_strict"].all())
    all_api = bool(readiness_df["ready_for_product_scoring_api"].all())

    print("All stages ready for notebook 5 strict business flow:", all_strict)
    print("All stages ready for product scoring API (guardrail-aware):", all_api)

    if all_strict:
        print(
            "[OK] You can move to notebook 5 with stricter business policy exploration.\n"
            "- Use score_raw / score_300_900 for ranking\n"
            "- Build cutoffs / policy banding on validation output\n"
            "- Only convert to PD after a separate calibration step"
        )
    elif all_api:
        print(
            "[OK WITH GUARDRAILS] Product scoring flow can move forward, but not every stage is strict-business-ready.\n"
            "- Use policy_mode by stage to control UI / workflow\n"
            "- full_decision: stage can support stronger cutoff experiments\n"
            "- starter_offer_only (Stage A): use for starter-loan recommendation / onboarding support\n"
            "- guardrail_only/manual_review_only: show score + reason codes + manual review recommendation\n"
            "- Keep score_proba out of customer-facing PD language"
        )
    else:
        print(
            "[HOLD] At least one stage is too weak even for guarded product scoring.\n"
            "Review stage_business_readiness.csv, stage_product_policy_summary.csv and product_scoring_manifest.json first."
        )


All stages ready for notebook 5 strict business flow: False
All stages ready for product scoring API (guardrail-aware): True
[OK WITH GUARDRAILS] Product scoring flow can move forward, but not every stage is strict-business-ready.
- Use policy_mode by stage to control UI / workflow
- full_decision: stage can support stronger cutoff experiments
- starter_offer_only (Stage A): use for starter-loan recommendation / onboarding support
- guardrail_only/manual_review_only: show score + reason codes + manual review recommendation
- Keep score_proba out of customer-facing PD language
